# 02 — Preprocessing & Augmentation Pipeline Demo

This notebook demonstrates the image preprocessing and augmentation pipelines
defined in `src/transforms.py`.

- Side-by-side comparison of train vs val transforms
- Visualise multiple augmented versions of the same image
- Pixel-value statistics after normalisation

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Add src to path
sys.path.insert(0, str(Path('..') / 'src'))

from transforms import get_train_transforms, get_val_test_transforms

plt.rcParams['figure.dpi'] = 120

## 1. Load a sample image

In [ ]:
# Replace with an actual image path once data is downloaded
SAMPLE_IMAGE_PATH = '../data/raw/ISIC_2019_Training_Input/ISIC_0024306.jpg'

pil_img = Image.open(SAMPLE_IMAGE_PATH).convert('RGB')
img_array = np.array(pil_img)

plt.figure(figsize=(5, 5))
plt.imshow(pil_img)
plt.title(f'Original image  {img_array.shape}')
plt.axis('off')
plt.show()

## 2. Compare train vs val/test transforms

In [ ]:
IMAGE_SIZE = 300

train_tfm = get_train_transforms(IMAGE_SIZE)
val_tfm   = get_val_test_transforms(IMAGE_SIZE)

# Apply once
train_result = train_tfm(image=img_array)['image']  # CHW tensor
val_result   = val_tfm(image=img_array)['image']

def tensor_to_display(t):
    """Denormalise a CHW tensor for display (approx ImageNet stats)."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = t.numpy().transpose(1, 2, 0)
    img  = img * std + mean
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(pil_img.resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[0].set_title('Original (resized)')
axes[1].imshow(tensor_to_display(train_result))
axes[1].set_title('Train transform')
axes[2].imshow(tensor_to_display(val_result))
axes[2].set_title('Val/Test transform')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/results/transform_comparison.png', dpi=150)
plt.show()

## 3. Multiple augmented views of the same image

In [ ]:
N = 8  # number of augmented samples
cols = 4
rows = N // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))

for i, ax in enumerate(axes.flat):
    aug = train_tfm(image=img_array)['image']
    ax.imshow(tensor_to_display(aug))
    ax.set_title(f'Aug #{i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('Training Augmentation Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/results/augmentation_samples.png', dpi=150)
plt.show()

## 4. Pixel statistics after normalisation

In [ ]:
val_t = val_tfm(image=img_array)['image'].numpy()  # CHW float32

print('After val/test normalisation:')
for c, name in enumerate(['R', 'G', 'B']):
    ch = val_t[c]
    print(f'  Channel {name}: mean={ch.mean():.4f}, std={ch.std():.4f}, '
          f'min={ch.min():.4f}, max={ch.max():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
colors = ['red', 'green', 'blue']
for c, (ax, col) in enumerate(zip(axes, colors)):
    ax.hist(val_t[c].flatten(), bins=50, color=col, alpha=0.7)
    ax.set_title(f'Channel {colors[c].upper()} histogram')
    ax.set_xlabel('Normalised pixel value')
plt.tight_layout()
plt.show()